# 02 — Data Extraction

Run all four extractors to populate `data/raw/` with source data from openFDA.

In [ ]:
import logging
import sys

sys.path.insert(0, "..")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

from src.api import FDAClient
from src.extraction import (
    AdverseEventExtractor,
    ClassificationExtractor,
    ClearanceExtractor,
    RecallExtractor,
)

client = FDAClient()
print(f"API key configured: {bool(client.api_key)}")
print(f"Rate limit: {client.rate_limit} req/min")

## 1. Classification Extraction

In [ ]:
cls_extractor = ClassificationExtractor(client)
cls_result = cls_extractor.extract()
print(f"Classification: {cls_result}")

## 2. Recall Extraction

In [ ]:
recall_extractor = RecallExtractor(client)
recall_result = recall_extractor.extract()
print("\nRecall summary:")
for year, info in recall_result.get("years", {}).items():
    print(f"  {year}: {info['records']:,} records")
print(f"  Total: {recall_result['total_records']:,} records")

## 3. 510(k) Clearance Extraction

In [ ]:
k510_extractor = ClearanceExtractor(client)
k510_result = k510_extractor.extract()
print("\n510(k) summary:")
for year, info in k510_result.get("years", {}).items():
    print(f"  {year}: {info['records']:,} records")
print(f"  Total: {k510_result['total_records']:,} records")

## 4. Adverse Event Bulk Extraction

This downloads ZIP files from the openFDA bulk download endpoint. This may take a while depending on network speed.

In [ ]:
ae_extractor = AdverseEventExtractor(client)
ae_result = ae_extractor.extract(method="bulk")
print(f"\nAdverse events: {ae_result.get('total_files', 0)} files downloaded")

## 5. Validation — Verify data/raw/ Contents

In [ ]:
from src.config import DATA_RAW


def summarize_dir(path, pattern="*"):
    """Count files and total size in a directory."""
    files = list(path.rglob(pattern))
    files = [f for f in files if f.is_file() and not f.name.startswith("_") and f.name != ".gitkeep"]
    total_size = sum(f.stat().st_size for f in files)
    return len(files), total_size / 1_048_576  # MB


dirs = {
    "classification": DATA_RAW / "classification",
    "recalls": DATA_RAW / "recalls",
    "clearances": DATA_RAW / "clearances",
    "adverse_events": DATA_RAW / "adverse_events",
}

print(f"{'Source':<20} {'Files':>6} {'Size (MB)':>10}")
print("-" * 40)
for name, path in dirs.items():
    if path.exists():
        count, size = summarize_dir(path)
        print(f"{name:<20} {count:>6} {size:>10.1f}")
    else:
        print(f"{name:<20} {'N/A':>6} {'N/A':>10}")

## Next Steps

Phase 2 will:
1. Parse and flatten adverse event ZIPs into structured parquet
2. Clean classification dimension table
3. Standardize manufacturer names with fuzzy matching
4. Build QA framework for data quality checks